# Exoplanet Transit Light Curve
### ADAPT Workshop — Day 3
**Philippine Space Agency × CVIF-RCTP**

---

This notebook processes the photometry CSV file from Afterglow and produces a differential light curve showing an exoplanet transit dip using an ensemble of reference stars.

**Instructions:** Run each cell in order by pressing `Shift + Enter`. Do not skip any cell.

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from google.colab import files

print('Libraries loaded successfully.')

## Step 2: Upload Your CSV File
Run this cell, then click **Choose Files** and select the CSV file you downloaded from Afterglow.

In [ ]:
uploaded = files.upload()
filename = list(uploaded.keys())[0]
print('File uploaded:', filename)

## Step 3: Load and Preview the Data

In [ ]:
import io

df = pd.read_csv(io.BytesIO(uploaded[filename]))

print('Total rows:', len(df))
print('Sources found:', df['id'].unique().tolist())
print()
df[['id', 'jd', 'flux', 'flux_error', 'mag']].head(10)

## Step 4: Assign Host Star and Reference Stars

- The **host star** is the star with the transiting exoplanet — usually the brightest source in your CSV.
- The **reference stars** are nearby stable stars used to cancel out atmospheric effects.

> **Important:** Check the source IDs printed above and update the values below to match your CSV.
>
> To identify which source is your host star, look at the flux values — the host star is typically the brightest (highest flux).

In [ ]:
# -------------------------------------------------------
# UPDATE THESE VALUES TO MATCH YOUR CSV SOURCE IDs
HOST_STAR_ID    = 'SRC8'
REFERENCE_IDS   = ['SRC9', 'SRC10', 'SRC11']  # add or remove as needed
# -------------------------------------------------------

host = df[df['id'] == HOST_STAR_ID].copy().reset_index(drop=True)
t0 = host['jd'].min()
host['time_hrs'] = (host['jd'] - t0) * 24

print(f'Host star ({HOST_STAR_ID}): {len(host)} frames')
for rid in REFERENCE_IDS:
    sub = df[df['id'] == rid]
    print(f'Reference ({rid}):  {len(sub)} frames')

time_span = (host['jd'].max() - host['jd'].min()) * 24
print(f'\nObservation span: {time_span:.2f} hours')

## Step 5: Check Reference Star Stability

Each reference star's flux should stay roughly constant (flat line) throughout the observation.

- **Variation below 5%** — good reference star
- **Variation above 5%** — star may be variable, or atmospheric conditions were poor

If all reference stars show similar variation percentages, the cause is likely atmospheric and not the stars themselves.

In [ ]:
fig, axes = plt.subplots(len(REFERENCE_IDS), 1,
                         figsize=(10, 3 * len(REFERENCE_IDS)),
                         sharex=True)

if len(REFERENCE_IDS) == 1:
    axes = [axes]

for ax, rid in zip(axes, REFERENCE_IDS):
    sub = df[df['id'] == rid].copy()
    sub['time_hrs'] = (sub['jd'] - t0) * 24
    norm = sub['flux'] / sub['flux'].median()
    variation = sub['flux'].std() / sub['flux'].mean() * 100

    ax.plot(sub['time_hrs'], norm, 'o', markersize=3, label=rid)
    ax.axhline(1.0, color='gray', linestyle='--', linewidth=1)
    ax.set_ylabel('Normalized Flux')
    ax.set_title(f'{rid} — Variation: {variation:.2f}%')
    ax.legend()
    ax.grid(True, alpha=0.3)
    print(f'{rid} flux variation: {variation:.2f}%')

axes[-1].set_xlabel('Time from Start (hours)')
plt.suptitle('Reference Star Stability Check', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## Step 6: Build the Ensemble Reference

Instead of using one reference star, we combine all reference stars by summing their fluxes. This averages out the individual noise of each star, giving a more stable baseline.

```
Flux Ratio = Host Star Flux / (Ref1 Flux + Ref2 Flux + Ref3 Flux)
```

In [ ]:
merged = host[['jd', 'time_hrs', 'flux', 'flux_error']].copy()
merged.columns = ['jd', 'time_hrs', 'flux_host', 'flux_error_host']

for i, rid in enumerate(REFERENCE_IDS):
    sub = df[df['id'] == rid][['jd', 'flux', 'flux_error']].copy()
    sub.columns = ['jd', f'flux_ref{i+1}', f'flux_error_ref{i+1}']
    merged = pd.merge(merged, sub, on='jd')

# Sum all reference fluxes
ref_cols = [f'flux_ref{i+1}' for i in range(len(REFERENCE_IDS))]
merged['flux_ref_ensemble'] = merged[ref_cols].sum(axis=1)

# Compute flux ratio
merged['flux_ratio'] = merged['flux_host'] / merged['flux_ref_ensemble']

# Propagate uncertainty
err_ref_cols = [f'flux_error_ref{i+1}' for i in range(len(REFERENCE_IDS))]
merged['flux_ratio_err'] = merged['flux_ratio'] * np.sqrt(
    (merged['flux_error_host'] / merged['flux_host'])**2 +
    (merged[err_ref_cols].pow(2).sum(axis=1) / merged['flux_ref_ensemble']**2)
)

ens_var = merged['flux_ref_ensemble'].std() / merged['flux_ref_ensemble'].mean() * 100
print(f'Ensemble reference variation: {ens_var:.2f}%')
print(f'Frames in merged dataset:     {len(merged)}')

## Step 7: Remove Outliers
Frames more than 3 standard deviations from the median are removed.

In [ ]:
median = merged['flux_ratio'].median()
std    = merged['flux_ratio'].std()

clean = merged[
    (merged['flux_ratio'] >= median - 3 * std) &
    (merged['flux_ratio'] <= median + 3 * std)
].copy()

print(f'Frames removed: {len(merged) - len(clean)}')
print(f'Frames remaining: {len(clean)}')

## Step 8: Plot the Transit Light Curve

Look for a **dip** — a temporary drop in the flux ratio — where the exoplanet is crossing in front of its host star and blocking some of its light.

In [ ]:
baseline  = np.median(clean['flux_ratio'])
min_ratio = clean['flux_ratio'].min()
depth_pct = (baseline - min_ratio) / baseline * 100

fig, ax = plt.subplots(figsize=(11, 5))

ax.errorbar(
    clean['time_hrs'],
    clean['flux_ratio'],
    yerr=clean['flux_ratio_err'],
    fmt='o',
    color='steelblue',
    markersize=4,
    ecolor='lightsteelblue',
    elinewidth=1,
    capsize=2,
    label=f'Host / Ensemble Reference ({len(REFERENCE_IDS)} stars)'
)

ax.axhline(baseline, color='gray', linestyle='--',
           linewidth=1, label=f'Baseline ({baseline:.4f})')

ax.set_xlabel('Time from Start (hours)', fontsize=12)
ax.set_ylabel('Flux Ratio (Host / Ensemble Reference)', fontsize=12)
ax.set_title('Exoplanet Transit Light Curve', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Baseline flux ratio:        {baseline:.4f}')
print(f'Minimum flux ratio:         {min_ratio:.4f}')
print(f'Estimated dip depth:        {depth_pct:.2f}%')
print(f'Reference stars used:       {len(REFERENCE_IDS)}')
print()
print('Note: Open filter + atmospheric variability limits depth accuracy.')
print('The known transit depth of WASP-18b is ~1.2%.')

## Step 9: Save the Plot

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

ax.errorbar(
    clean['time_hrs'],
    clean['flux_ratio'],
    yerr=clean['flux_ratio_err'],
    fmt='o',
    color='steelblue',
    markersize=4,
    ecolor='lightsteelblue',
    elinewidth=1,
    capsize=2,
    label=f'Host / Ensemble Reference ({len(REFERENCE_IDS)} stars)'
)

ax.axhline(baseline, color='gray', linestyle='--',
           linewidth=1, label=f'Baseline ({baseline:.4f})')

ax.set_xlabel('Time from Start (hours)', fontsize=12)
ax.set_ylabel('Flux Ratio (Host / Ensemble Reference)', fontsize=12)
ax.set_title('Exoplanet Transit Light Curve', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('transit_lightcurve.png', dpi=150)
plt.show()

files.download('transit_lightcurve.png')
print('Saved and downloaded as transit_lightcurve.png')